# 03 — LLM Scoring

Run schizotypal trait scoring on all collected posts.

In [ ]:
import asyncio
import polars as pl
from social_llm.scoring.pipeline import score_dataset

In [ ]:
SEED_POST_ID = "REPLACE_ME"

# Load the collected posts
posts_df = pl.read_parquet(f"data/processed/network_{SEED_POST_ID}.parquet")
print(f"{len(posts_df)} posts to score")

In [ ]:
# Score all posts (incremental — safe to re-run)
scored_df = await score_dataset(posts_df, output_name=f"scored_{SEED_POST_ID}")

In [ ]:
scored_df.head()

In [ ]:
# Sanity check — look at some high-scoring posts
trait_cols = ["magical_thinking", "ideas_of_reference", "unusual_perceptions",
             "paranoid_ideation", "odd_speech", "social_anxiety"]

scored_df = scored_df.with_columns(
    pl.sum_horizontal(trait_cols).alias("total_score")
)

high = scored_df.sort("total_score", descending=True).head(10)
for row in high.to_dicts():
    print(f"Score {row['total_score']} | @{row['username']}")
    print(f"  {row['text'][:200]}")
    print(f"  Reasoning: {row['reasoning']}")
    print()

In [ ]:
# Cannabis mentions
cannabis_posts = scored_df.filter(pl.col("cannabis_mention"))
print(f"{len(cannabis_posts)} posts mention cannabis")
for row in cannabis_posts.head(10).to_dicts():
    print(f"@{row['username']}: {row['text'][:200]}")
    print(f"  Context: {row['cannabis_context']}")
    print()